# Step 0 - Preflight (pilot smoke test)
Runtime > Change runtime type > **T4 GPU** (not TPU). Run cell 1 (install), **restart the session**, then run the remaining cells in order and paste the REPORT block back to Claude. No secrets are printed; HF auth (if needed) comes from the `HF_TOKEN` Colab secret.

In [ ]:
# RUN THIS CELL FIRST, THEN Runtime > Restart session, THEN run everything below.
# (Installing after numpy/torch are imported corrupts the kernel; numpy is deliberately not upgraded here.)
!pip -q install -U transformers accelerate qwen-omni-utils soundfile pillow


In [ ]:
# After restart: confirm imports are healthy before loading the model
import numpy, scipy, sklearn, transformers
print('numpy',numpy.__version__,'| scipy',scipy.__version__,'| sklearn',sklearn.__version__,'| transformers',transformers.__version__)
from transformers import GenerationMixin
try:
    import flash_attn; print('flash_attn present', flash_attn.__version__)
except Exception as e:
    print('flash_attn NOT available (expected on T4):', type(e).__name__)


In [ ]:
import subprocess, torch, os, time, json
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv'],capture_output=True,text=True).stdout)
props=torch.cuda.get_device_properties(0)
print('torch',torch.__version__,'| cuda',torch.version.cuda,'| bf16 supported:',torch.cuda.is_bf16_supported(),'| compute cap',props.major,props.minor)

In [ ]:
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
MID='Qwen/Qwen2.5-Omni-3B'
t0=time.time()
model=Qwen2_5OmniForConditionalGeneration.from_pretrained(MID,torch_dtype=torch.float16,device_map='auto',attn_implementation='sdpa')
processor=Qwen2_5OmniProcessor.from_pretrained(MID)
print('load seconds',round(time.time()-t0,1))
print('after load: allocated GB',round(torch.cuda.memory_allocated()/2**30,2))
print('top-level children:',[n for n,_ in model.named_children()])

In [ ]:
# Free the speech-output Talker if the API allows (we only need text out). Log whether it worked.
try:
    model.disable_talker(); print('talker disabled')
except Exception as e:
    print('disable_talker unavailable/failed:',type(e).__name__,e)
torch.cuda.empty_cache(); print('allocated GB',round(torch.cuda.memory_allocated()/2**30,2))

In [ ]:
import numpy as np
from PIL import Image
from qwen_omni_utils import process_mm_info
SYS="You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech."
Image.fromarray((np.random.rand(224,224,3)*255).astype('uint8')).save('/content/t.jpg')
import soundfile as sf
sr=16000; tt=np.arange(sr*2)/sr; sf.write('/content/t.wav',(0.3*np.sin(2*np.pi*440*tt)).astype('float32'),sr)
cases={'text':[{'type':'text','text':'Say hello in one sentence.'}],
       'image':[{'type':'image','image':'/content/t.jpg'},{'type':'text','text':'Describe this image briefly.'}],
       'audio':[{'type':'audio','audio':'/content/t.wav'},{'type':'text','text':'What do you hear?'}]}
report={}
for k,content in cases.items():
    conv=[{'role':'system','content':[{'type':'text','text':SYS}]},{'role':'user','content':content}]
    try:
        torch.cuda.reset_peak_memory_stats()
        text=processor.apply_chat_template(conv,add_generation_prompt=True,tokenize=False)
        a,i,v=process_mm_info(conv,use_audio_in_video=False)
        inp=processor(text=text,audio=a,images=i,videos=v,return_tensors='pt',padding=True,use_audio_in_video=False).to(model.device).to(model.dtype)
        with torch.no_grad():
            out=model.generate(**inp,max_new_tokens=20,return_audio=False,use_audio_in_video=False)
        ids=out[0] if isinstance(out,tuple) else out
        s=processor.batch_decode(ids,skip_special_tokens=True)[0][-120:]
        report[k]={'ok':True,'peak_GB':round(torch.cuda.max_memory_allocated()/2**30,2),'sample':s}
    except Exception as e:
        report[k]={'ok':False,'error':f'{type(e).__name__}: {str(e)[:300]}'}
    print(k,report[k])

In [ ]:
print('===== REPORT (paste to Claude) =====')
print(json.dumps({'gpu':torch.cuda.get_device_name(0),'total_GB':round(props.total_memory/2**30,2),
 'torch':torch.__version__,'transformers':transformers.__version__,'bf16':torch.cuda.is_bf16_supported(),
 'allocated_after_tests_GB':round(torch.cuda.memory_allocated()/2**30,2),'results':report},indent=1))